In [1]:
# ============================================================
# Cell 1: Install Requirements
# ============================================================
!pip -q install timm torchmetrics albumentations opencv-python-headless scikit-learn seaborn tqdm open-clip-torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00


In [2]:
# ============================================================
# Cell 2: Mount Drive & Cache Dataset
# ============================================================
import os
import shutil
import time
from pathlib import Path
from google.colab import drive
import pandas as pd

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/AI_Image_Detection")
PROCESSED_DATASET = PROJECT_DIR / "processed_dataset_final"

if not PROCESSED_DATASET.exists():
    raise FileNotFoundError(f"Dataset not found: {PROCESSED_DATASET}")

# LOCAL_DATASET_ROOT = Path("/content/local_dataset")
# LOCAL_DATASET = LOCAL_DATASET_ROOT / "processed_dataset_final"

# if not LOCAL_DATASET.exists():
#     print("=" * 70)
#     print("Creating local dataset cache (Google Drive -> Colab SSD)...")
#     print("=" * 70)
#     start = time.time()
#     shutil.copytree(PROCESSED_DATASET, LOCAL_DATASET)
#     print(f"Dataset cached successfully in {time.time() - start:.2f} seconds.\n")
# else:
#     print("=" * 70)
#     print("Local dataset cache already exists. Skipping copy.")
#     print("=" * 70)

# # Map paths to local SSD
# DATASET_ROOT = LOCAL_DATASET


# Use dataset directly from Google Drive
DATASET_ROOT = PROCESSED_DATASET


TRAIN_DIR = DATASET_ROOT / "train"
VAL_DIR = DATASET_ROOT / "val"
TEST_DIR = DATASET_ROOT / "test"

TRAIN_MANIFEST = DATASET_ROOT / "train_manifest.csv"
VAL_MANIFEST = DATASET_ROOT / "val_manifest.csv"
TEST_MANIFEST = DATASET_ROOT / "test_manifest.csv"

# Persistent Output Paths (Save back to Drive so we don't lose them on disconnect)
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints_final"
BEST_MODEL_DIR = PROJECT_DIR / "models_final"
LOG_DIR        = PROJECT_DIR / "logs_final"
RESULTS_DIR    = PROJECT_DIR / "results_final"
REPORT_DIR     = PROJECT_DIR / "reports_final"

for directory in [CHECKPOINT_DIR, BEST_MODEL_DIR, LOG_DIR, RESULTS_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Environment Paths Initialized successfully.")


Mounted at /content/drive
Environment Paths Initialized successfully.


In [3]:
# ============================================================
# Cell 3: Imports & Configuration
# ============================================================
import csv
import json
import random
import copy
from dataclasses import dataclass, asdict
from typing import Optional, Dict, Tuple

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import open_clip
import torchvision.models as models
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
    accuracy_score, f1_score, balanced_accuracy_score, matthews_corrcoef,
    confusion_matrix, classification_report, brier_score_loss
)
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import seaborn as sns

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

@dataclass
class TrainConfig:
    # --- Auto-Resume ---
    resume_from_checkpoint: bool = True

    # --- Directories ---
    logs_dir: str = str(LOG_DIR)
    results_dir: str = str(RESULTS_DIR)
    reports_dir: str = str(REPORT_DIR)
    checkpoint_dir: str = str(CHECKPOINT_DIR)

    # --- Data & Reproducibility ---
    batch_size: int = 16 # Lowered to 16 to prevent OOM with ViT-L/14 on Colab T4
    num_workers: int = 2
    seed: int = 42

    # --- Architecture & Regularization ---
    hidden1: int = 512
    hidden2: int = 256
    feature_dropout: float = 0.1
    fc_dropout: float = 0.4

    # --- Loss Parameters ---
    focal_gamma: float = 2.0

    # --- Stage 1: Linear Probe ---
    stage1_epochs: int = 10
    stage1_lr: float = 2e-4
    stage1_label_smoothing: float = 0.08
    stage1_patience: int = 4
    stage1_weight_decay: float = 1e-2

    # --- Stage 2: End-to-End Fine-Tuning ---
    stage2_epochs: int = 10
    stage2_head_lr: float = 5e-5
    stage2_backbone_lr_mult: float = 0.1
    stage2_weight_decay: float = 1e-2
    stage2_backbone_weight_decay: float = 1e-4
    stage2_patience: int = 3
    stage2_grad_clip: float = 1.0
    stage2_ema_decay: float = 0.999
    stage2_label_smoothing: float = 0.05

    # --- Training Flags ---
    use_weighted_sampler: bool = True
    use_amp: bool = True
    use_tta_flip: bool = True

cfg = TrainConfig()
set_seed(cfg.seed)
with open(os.path.join(cfg.reports_dir, "config.json"), "w") as f:
    json.dump(asdict(cfg), f, indent=4)


In [4]:
# ============================================================
# Cell 4: Dataset & Augmentations
# ============================================================
class FaceAuthenticityDataset(Dataset):
    def __init__(self, manifest: list, image_dir: Path, train: bool = True):
        self.manifest = manifest
        self.image_dir = image_dir
        self.train = train

        # We output to [0,1] range because the Model's Backbone classes apply
        # their own specific ImageNet/CLIP mean & std internally.
        self.base_transform = A.Compose([
            A.Resize(384, 384),
            ToTensorV2()
        ])

        self.train_transform = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.ImageCompression(quality_lower=70, quality_upper=100, p=0.3),
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
            A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05, p=0.3),
            A.Resize(384, 384),
            ToTensorV2()
        ])

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, idx):
        row = self.manifest[idx]

        # Handle different column names (filename vs image_path)
        img_name = row.get('filename', row.get('image_path', None))
        if img_name is None:
            img_name = list(row.values())[0] # Fallback to first column

        img_path = self.image_dir / img_name

        # Load Image
        image = cv2.imread(str(img_path))
        if image is None:
            # Fallback to zeros if image is corrupted to prevent crashing
            image = np.zeros((384, 384, 3), dtype=np.uint8)
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Apply transforms
        if self.train:
            augmented = self.train_transform(image=image)
        else:
            augmented = self.base_transform(image=image)

        # Normalize to [0, 1] range for the model's internal standardization buffers
        image_tensor = augmented['image'].float() / 255.0

        # Label: Assuming 'ai' or 'fake' is 1, 'real' is 0
        label_str = str(row.get('label', '0')).lower()
        label = 1.0 if label_str in ['ai', 'fake', '1'] else 0.0

        source = str(row.get('source', 'unknown'))
        return image_tensor, torch.tensor(label, dtype=torch.float32), source


In [5]:
# ============================================================
# Cell 5: Model Architecture
# ============================================================
class FrozenFaceBackbone(nn.Module):
    embedding_dim = 512
    def __init__(self):
        super().__init__()
        # Fallback to robust torchvision ResNet50
        self.model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.model.fc = nn.Linear(self.model.fc.in_features, self.embedding_dim)

        self.model.eval()
        for p in self.model.parameters(): p.requires_grad_(False)

        # Default Face Mean/Std
        self.register_buffer("mean", torch.tensor([0.5, 0.5, 0.5]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.5, 0.5, 0.5]).view(1, 3, 1, 1))

    def forward(self, x01):
        x = (x01 - self.mean) / self.std
        emb = self.model(x)
        return F.normalize(emb, dim=1)


class FrozenCLIPBackbone(nn.Module):
    embedding_dim = 768
    def __init__(self):
        super().__init__()
        model, _, _ = open_clip.create_model_and_transforms('ViT-L-14', pretrained='openai')
        self.model = model.visual
        self.model.eval()
        for p in self.model.parameters(): p.requires_grad_(False)

        # CLIP standard mean/std
        clip_mean = [0.48145466, 0.4578275, 0.40821073]
        clip_std = [0.26862954, 0.26130258, 0.27577711]
        self.register_buffer("mean", torch.tensor(clip_mean).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(clip_std).view(1, 3, 1, 1))

    def forward(self, x01):
        x = (x01 - self.mean) / self.std
        feat = self.model(x)
        return F.normalize(feat.float(), dim=1)


class FusionHead(nn.Module):
    def __init__(self, face_dim, clip_dim, hidden1, hidden2, feat_drop, fc_drop):
        super().__init__()
        self.input_norm = nn.LayerNorm(face_dim + clip_dim)
        self.feature_dropout = nn.Dropout(feat_drop)
        self.net = nn.Sequential(
            nn.Linear(face_dim + clip_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.GELU(),
            nn.Dropout(fc_drop),
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.GELU(),
            nn.Dropout(fc_drop),
            nn.Linear(hidden2, 1),
        )

    def forward(self, face_emb, clip_emb):
        x = torch.cat([face_emb, clip_emb], dim=1)
        return self.net(self.feature_dropout(self.input_norm(x))).squeeze(1)


class FusionModel(nn.Module):
    def __init__(self, cfg: TrainConfig):
        super().__init__()
        self.face_backbone = FrozenFaceBackbone()
        self.clip_backbone = FrozenCLIPBackbone()
        self.head = FusionHead(
            self.face_backbone.embedding_dim,
            self.clip_backbone.embedding_dim,
            cfg.hidden1, cfg.hidden2, cfg.feature_dropout, cfg.fc_dropout
        )
        self._bn_frozen_modules = []

    def forward(self, x_384):
        # Prepare CLIP Input (224x224)
        clip_input = F.interpolate(x_384, size=(224, 224), mode='bicubic', antialias=True)

        # Prepare Face Crop (Central 80% resized to 112x112)
        crop_size = int(x_384.shape[-1] * 0.8)
        start = (x_384.shape[-1] - crop_size) // 2
        face_crop = x_384[:, :, start:start + crop_size, start:start + crop_size]
        face_input = F.interpolate(face_crop, size=(112, 112), mode='bicubic', antialias=True)

        return self.head(self.face_backbone(face_input), self.clip_backbone(clip_input))

    def unfreeze_last_blocks(self):
        trainable_params, bn_modules = [], []

        # Unfreeze CLIP last block
        try:
            last_clip_block = self.clip_backbone.model.transformer.resblocks[-1]
            for p in last_clip_block.parameters():
                p.requires_grad_(True)
                trainable_params.append(p)
            if hasattr(self.clip_backbone.model, 'ln_post'):
                for p in self.clip_backbone.model.ln_post.parameters():
                    p.requires_grad_(True)
                    trainable_params.append(p)
            self.clip_backbone.model.train()
        except Exception as e:
            print(f"[Warning] CLIP unfreeze failed: {e}")

        # Unfreeze ResNet last layer
        try:
            last_face_layer = self.face_backbone.model.layer4
            for p in last_face_layer.parameters():
                p.requires_grad_(True)
                trainable_params.append(p)
            for m in last_face_layer.modules():
                if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                    bn_modules.append(m)
            self.face_backbone.model.train()
        except Exception as e:
            print(f"[Warning] Face Backbone unfreeze failed: {e}")

        self._bn_frozen_modules = bn_modules
        self._freeze_bn_stats()
        return trainable_params

    def _freeze_bn_stats(self):
        for m in self._bn_frozen_modules: m.eval()

    def train(self, mode: bool = True):
        super().train(mode)
        if mode: self._freeze_bn_stats()
        return self

class ModelEMA:
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters(): p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model: nn.Module):
        msd = model.state_dict()
        for k, v in self.ema.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(self.decay).add_(msd[k].detach(), alpha=1 - self.decay)
            else:
                v.copy_(msd[k])


In [6]:
# ============================================================
# Cell 5: Model Architecture
# ============================================================
class FrozenFaceBackbone(nn.Module):
    embedding_dim = 512
    def __init__(self):
        super().__init__()
        # Fallback to robust torchvision ResNet50
        self.model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.model.fc = nn.Linear(self.model.fc.in_features, self.embedding_dim)

        self.model.eval()
        for p in self.model.parameters(): p.requires_grad_(False)

        # Default Face Mean/Std
        self.register_buffer("mean", torch.tensor([0.5, 0.5, 0.5]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.5, 0.5, 0.5]).view(1, 3, 1, 1))

    def forward(self, x01):
        x = (x01 - self.mean) / self.std
        emb = self.model(x)
        return F.normalize(emb, dim=1)


class FrozenCLIPBackbone(nn.Module):
    embedding_dim = 768
    def __init__(self):
        super().__init__()
        model, _, _ = open_clip.create_model_and_transforms('ViT-L-14', pretrained='openai')
        self.model = model.visual
        self.model.eval()
        for p in self.model.parameters(): p.requires_grad_(False)

        # CLIP standard mean/std
        clip_mean = [0.48145466, 0.4578275, 0.40821073]
        clip_std = [0.26862954, 0.26130258, 0.27577711]
        self.register_buffer("mean", torch.tensor(clip_mean).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(clip_std).view(1, 3, 1, 1))

    def forward(self, x01):
        x = (x01 - self.mean) / self.std
        feat = self.model(x)
        return F.normalize(feat.float(), dim=1)


class FusionHead(nn.Module):
    def __init__(self, face_dim, clip_dim, hidden1, hidden2, feat_drop, fc_drop):
        super().__init__()
        self.input_norm = nn.LayerNorm(face_dim + clip_dim)
        self.feature_dropout = nn.Dropout(feat_drop)
        self.net = nn.Sequential(
            nn.Linear(face_dim + clip_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.GELU(),
            nn.Dropout(fc_drop),
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.GELU(),
            nn.Dropout(fc_drop),
            nn.Linear(hidden2, 1),
        )

    def forward(self, face_emb, clip_emb):
        x = torch.cat([face_emb, clip_emb], dim=1)
        return self.net(self.feature_dropout(self.input_norm(x))).squeeze(1)


class FusionModel(nn.Module):
    def __init__(self, cfg: TrainConfig):
        super().__init__()
        self.face_backbone = FrozenFaceBackbone()
        self.clip_backbone = FrozenCLIPBackbone()
        self.head = FusionHead(
            self.face_backbone.embedding_dim,
            self.clip_backbone.embedding_dim,
            cfg.hidden1, cfg.hidden2, cfg.feature_dropout, cfg.fc_dropout
        )
        self._bn_frozen_modules = []

    def forward(self, x_384):
        # Prepare CLIP Input (224x224)
        clip_input = F.interpolate(x_384, size=(224, 224), mode='bicubic', antialias=True)

        # Prepare Face Crop (Central 80% resized to 112x112)
        crop_size = int(x_384.shape[-1] * 0.8)
        start = (x_384.shape[-1] - crop_size) // 2
        face_crop = x_384[:, :, start:start + crop_size, start:start + crop_size]
        face_input = F.interpolate(face_crop, size=(112, 112), mode='bicubic', antialias=True)

        return self.head(self.face_backbone(face_input), self.clip_backbone(clip_input))

    def unfreeze_last_blocks(self):
        trainable_params, bn_modules = [], []

        # Unfreeze CLIP last block
        try:
            last_clip_block = self.clip_backbone.model.transformer.resblocks[-1]
            for p in last_clip_block.parameters():
                p.requires_grad_(True)
                trainable_params.append(p)
            if hasattr(self.clip_backbone.model, 'ln_post'):
                for p in self.clip_backbone.model.ln_post.parameters():
                    p.requires_grad_(True)
                    trainable_params.append(p)
            self.clip_backbone.model.train()
        except Exception as e:
            print(f"[Warning] CLIP unfreeze failed: {e}")

        # Unfreeze ResNet last layer
        try:
            last_face_layer = self.face_backbone.model.layer4
            for p in last_face_layer.parameters():
                p.requires_grad_(True)
                trainable_params.append(p)
            for m in last_face_layer.modules():
                if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                    bn_modules.append(m)
            self.face_backbone.model.train()
        except Exception as e:
            print(f"[Warning] Face Backbone unfreeze failed: {e}")

        self._bn_frozen_modules = bn_modules
        self._freeze_bn_stats()
        return trainable_params

    def _freeze_bn_stats(self):
        for m in self._bn_frozen_modules: m.eval()

    def train(self, mode: bool = True):
        super().train(mode)
        if mode: self._freeze_bn_stats()
        return self

class ModelEMA:
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters(): p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model: nn.Module):
        msd = model.state_dict()
        for k, v in self.ema.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(self.decay).add_(msd[k].detach(), alpha=1 - self.decay)
            else:
                v.copy_(msd[k])


In [7]:
# ============================================================
# Cell 6: Utilities & Checkpointing
# ============================================================
import os
import torch
import torch.nn as nn

def save_checkpoint(path, model, optimizer, scheduler, scaler, ema, epoch, stage, best_auc, best_ckpt_path, history, early_stopping_counter):
    """Saves the exact state of training to resume later if interrupted."""
    state = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'ema_state_dict': ema.ema.state_dict() if ema else None,
        'epoch': epoch,
        'stage': stage,
        'best_auc': best_auc,
        'best_ckpt_path': best_ckpt_path,  # <--- FIX: Now saving the path
        'history': history,
        'early_stopping_counter': early_stopping_counter
    }
    torch.save(state, path)

class DescriptiveEarlyStopping:
    def __init__(self, patience=4, min_delta=0.001, mode='max', save_dir='checkpoints', prefix='model'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.save_dir = save_dir
        self.prefix = prefix
        self.best_score = -float('inf') if mode == 'max' else float('inf')
        self.counter = 0
        self.best_ckpt_path = None

    def __call__(self, current_score, epoch, model):
        improved = (current_score > (self.best_score + self.min_delta)) if self.mode == 'max' \
            else (current_score < (self.best_score - self.min_delta))

        if improved:
            self.best_score = current_score
            self.counter = 0
            ckpt_name = f"{self.prefix}_epoch_{epoch:02d}_auc_{current_score:.4f}.pt"
            self.best_ckpt_path = os.path.join(self.save_dir, ckpt_name)

            for f in os.listdir(self.save_dir):
                if f.startswith(self.prefix) and f.endswith(".pt"):
                    os.remove(os.path.join(self.save_dir, f))

            torch.save({"model_state_dict": model.state_dict()}, self.best_ckpt_path)
            return False

        self.counter += 1
        if self.counter >= self.patience:
            print(f"-> Early stopping triggered (Patience {self.patience} reached).")
            return True
        return False

    def restore_best_weights(self, model):
        if self.best_ckpt_path and os.path.exists(self.best_ckpt_path):
            ckpt = torch.load(self.best_ckpt_path, map_location=DEVICE)
            model.load_state_dict(ckpt["model_state_dict"])
            print(f"-> Restored checkpoint: {os.path.basename(self.best_ckpt_path)}")


class FocalLossWithLogits(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, logits, targets):
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + 0.5 * self.label_smoothing
        bce_loss = self.bce(logits, targets)
        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (alpha_t * (1 - p_t) ** self.gamma * bce_loss).mean()

class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        return logits / self.temperature

    def fit(self, logits, labels):
        nll_criterion = nn.BCEWithLogitsLoss()
        optimizer = torch.optim.LBFGS([self.temperature], lr=0.01, max_iter=50)
        def eval():
            optimizer.zero_grad()
            loss = nll_criterion(self.forward(logits), labels)
            loss.backward()
            return loss
        optimizer.step(eval)
        return self.temperature.item()


In [8]:
# ============================================================
# Cell 7: Training Engine
# ============================================================
from tqdm.auto import tqdm

@torch.no_grad()
def collect_predictions(model, loader, use_tta_flip=True):
    model.eval()
    all_logits, all_labels = [], []
    for images, labels, _ in loader:
        images = images.to(DEVICE)
        logits = model(images)
        if use_tta_flip:
            logits = (logits + model(torch.flip(images, dims=[-1]))) / 2.0
        all_logits.append(logits.cpu())
        all_labels.append(labels)
    return torch.cat(all_logits), torch.cat(all_labels)

def train_stage(model, train_loader, val_loader, pos_weight, cfg: TrainConfig, stage=1, resume_state=None):
    is_stage1 = (stage == 1)
    epochs = cfg.stage1_epochs if is_stage1 else cfg.stage2_epochs
    prefix = "stage1" if is_stage1 else "stage2"

    # 1. Setup Optimizers & Loss
    if is_stage1:
        optimizer = torch.optim.AdamW(model.head.parameters(), lr=cfg.stage1_lr, weight_decay=cfg.stage1_weight_decay)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=DEVICE))
        lbl_smooth = cfg.stage1_label_smoothing
        ema = None
    else:
        backbone_params = model.unfreeze_last_blocks()
        param_groups = [{'params': model.head.parameters(), 'lr': cfg.stage2_head_lr, 'weight_decay': cfg.stage2_weight_decay}]
        if backbone_params:
            param_groups.append({'params': backbone_params, 'lr': cfg.stage2_head_lr * cfg.stage2_backbone_lr_mult, 'weight_decay': cfg.stage2_backbone_weight_decay})
        optimizer = torch.optim.AdamW(param_groups)
        alpha_val = pos_weight / (1 + pos_weight)
        loss_fn = FocalLossWithLogits(alpha=alpha_val, gamma=cfg.focal_gamma, label_smoothing=cfg.stage2_label_smoothing)
        lbl_smooth = 0.0
        ema = ModelEMA(model, decay=cfg.stage2_ema_decay)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=cfg.use_amp)

    early_stopper = DescriptiveEarlyStopping(
        patience=cfg.stage1_patience if is_stage1 else cfg.stage2_patience,
        save_dir=cfg.checkpoint_dir, prefix=prefix
    )

    # 2. Auto-Resume Logic
    start_epoch = 1
    history = []

    if resume_state is not None and resume_state.get('stage') == stage:
        print(f"--> Resuming {prefix} from Epoch {resume_state['epoch']}...")
        model.load_state_dict(resume_state['model_state_dict'])
        optimizer.load_state_dict(resume_state['optimizer_state_dict'])
        scheduler.load_state_dict(resume_state['scheduler_state_dict'])
        scaler.load_state_dict(resume_state['scaler_state_dict'])
        if ema and resume_state.get('ema_state_dict'):
            ema.ema.load_state_dict(resume_state['ema_state_dict'])

        start_epoch = resume_state['epoch'] + 1
        history = resume_state.get('history', [])

        # <--- FIX: Restoring the early stopper state completely
        early_stopper.best_score = resume_state.get('best_auc', -float('inf'))
        early_stopper.best_ckpt_path = resume_state.get('best_ckpt_path', None)
        early_stopper.counter = resume_state.get('early_stopping_counter', 0)

    # 3. Training Loop
    for epoch in range(start_epoch, epochs + 1):
        model.train()
        total_loss = 0.0

        for images, labels, _ in tqdm(train_loader, desc=f"{prefix.upper()} Epoch {epoch}/{epochs}", leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            targets = labels * (1 - lbl_smooth) + 0.5 * lbl_smooth if is_stage1 else labels

            with torch.cuda.amp.autocast(enabled=cfg.use_amp):
                logits = model(images)
                loss = loss_fn(logits, targets)

            optimizer.zero_grad()
            scaler.scale(loss).backward()

            if not is_stage1:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_([p for g in optimizer.param_groups for p in g['params']], cfg.stage2_grad_clip)

            scaler.step(optimizer)
            scaler.update()

            if ema: ema.update(model)
            total_loss += loss.item() * images.size(0)

        scheduler.step()
        train_loss = total_loss / len(train_loader.dataset)

        # Validation
        eval_model = model if is_stage1 else ema.ema
        val_logits, val_labels = collect_predictions(eval_model, val_loader, use_tta_flip=(not is_stage1) and cfg.use_tta_flip)

        probs = torch.sigmoid(val_logits).numpy()
        y_true = val_labels.numpy()
        val_auc = float(roc_auc_score(y_true, probs))
        val_pr = float(average_precision_score(y_true, probs))

        print(f"{prefix.upper()} | Ep {epoch:>2}/{epochs} | Loss: {train_loss:.4f} | AUC: {val_auc:.4f} | PR-AUC: {val_pr:.4f}")

        history.append({"epoch": epoch, "train_loss": train_loss, "val_auc": val_auc, "val_pr_auc": val_pr})

        stop_training = early_stopper(val_auc, epoch, eval_model)

        # <--- FIX: Passing early_stopper.best_ckpt_path to the save state
        save_checkpoint(
          os.path.join(cfg.checkpoint_dir, "latest_training_checkpoint.pt"),
          model, optimizer, scheduler, scaler, ema, epoch, stage,
          early_stopper.best_score, early_stopper.best_ckpt_path,
          history, early_stopper.counter
        )

        if stop_training: break

    early_stopper.restore_best_weights(model)
    return model


In [ ]:
# ============================================================
# Cell 8: Execution & Final Evaluation
# ============================================================
import os
import csv
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import classification_report, roc_auc_score

def load_manifest(path):
    with open(path) as f: return list(csv.DictReader(f))

train_manifest = load_manifest(TRAIN_MANIFEST)
val_manifest = load_manifest(VAL_MANIFEST)
test_manifest = load_manifest(TEST_MANIFEST) # FIX: Load Test Data

# Target Ratio & Weights
ai_count = sum(1 for row in train_manifest if str(row.get('label', '0')).lower() in ['ai', 'fake', '1'])
real_count = len(train_manifest) - ai_count
pos_weight = real_count / max(1, ai_count)
print(f"Dataset Ratio - Real: {real_count} | AI: {ai_count} -> pos_weight: {pos_weight:.3f}")

train_ds = FaceAuthenticityDataset(train_manifest, TRAIN_DIR, train=True)
val_ds = FaceAuthenticityDataset(val_manifest, VAL_DIR, train=False)
test_ds = FaceAuthenticityDataset(test_manifest, TEST_DIR, train=False) # FIX: Initialize Test Dataset

# Data Loader Performance optimizations (Prefetch & Persistent)
pin_memory = torch.cuda.is_available()
persistent = cfg.num_workers > 0
prefetch = 2 if persistent else None

# Initialize DataLoaders
if cfg.use_weighted_sampler:
    labels = [1 if str(row.get('label', '0')).lower() in ['ai', 'fake', '1'] else 0 for row in train_manifest]
    weights = [1.0/ai_count if l==1 else 1.0/real_count for l in labels]
    sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
    train_loader = DataLoader(
        train_ds, batch_size=cfg.batch_size, sampler=sampler,
        num_workers=cfg.num_workers, pin_memory=pin_memory,
        persistent_workers=persistent, prefetch_factor=prefetch, drop_last=True
    )
else:
    train_loader = DataLoader(
        train_ds, batch_size=cfg.batch_size, shuffle=True,
        num_workers=cfg.num_workers, pin_memory=pin_memory,
        persistent_workers=persistent, prefetch_factor=prefetch, drop_last=True
    )

val_loader = DataLoader(
    val_ds, batch_size=cfg.batch_size, shuffle=False,
    num_workers=cfg.num_workers, pin_memory=pin_memory,
    persistent_workers=persistent, prefetch_factor=prefetch
)

test_loader = DataLoader(
    test_ds, batch_size=cfg.batch_size, shuffle=False,
    num_workers=cfg.num_workers, pin_memory=pin_memory,
    persistent_workers=persistent, prefetch_factor=prefetch
)

# Build Model & Check Resume State
model = FusionModel(cfg).to(DEVICE)
resume_state = None
latest_ckpt = os.path.join(cfg.checkpoint_dir, "latest_training_checkpoint.pt")

if cfg.resume_from_checkpoint and os.path.exists(latest_ckpt):
    print(f"Found existing checkpoint. Loading {latest_ckpt}...")
    resume_state = torch.load(latest_ckpt, map_location=DEVICE)
    current_stage = resume_state.get('stage', 1)
else:
    current_stage = 1

# Execute Stages
if current_stage == 1:
    print("\n--- STAGE 1: Linear Probe ---")
    model = train_stage(model, train_loader, val_loader, pos_weight, cfg, stage=1, resume_state=resume_state)
    current_stage = 2
    resume_state = None

if current_stage == 2:
    print("\n--- STAGE 2: End-to-End Fine-Tune ---")
    model = train_stage(model, train_loader, val_loader, pos_weight, cfg, stage=2, resume_state=resume_state)

print("\n--- FINAL EVALUATION (TTA & Temperature Scaling) ---")

# FIX: Issue 2 & 3 - Learn on Validation, Report on Test
print("1. Calibrating Temperature on Validation Set...")
val_logits, val_labels = collect_predictions(model, val_loader, use_tta_flip=cfg.use_tta_flip)
scaler_model = TemperatureScaler().to(DEVICE)
optimal_temp = scaler_model.fit(val_logits.to(DEVICE), val_labels.float().to(DEVICE))
print(f"Optimal Temperature Calibrated: {optimal_temp:.4f}")

print("\n2. Evaluating on Unseen Test Set...")
test_logits, test_labels = collect_predictions(model, test_loader, use_tta_flip=cfg.use_tta_flip)

# Scale Test Predictions using Validation-learned Temperature
y_prob = torch.sigmoid(test_logits / optimal_temp).numpy()
y_true = test_labels.numpy()
preds = (y_prob >= 0.5).astype(int)

print("\n--- FINAL TEST REPORT ---")
print(classification_report(y_true, preds, target_names=['Real', 'AI']))
print(f"ROC-AUC: {roc_auc_score(y_true, y_prob):.4f}")

# Save Final Artifacts
torch.save(model.state_dict(), os.path.join(cfg.checkpoint_dir, "best_final_model.pt"))
print(f"\nArtifacts successfully saved to {cfg.checkpoint_dir}/")


Dataset Ratio - Real: 12803 | AI: 10214 -> pos_weight: 1.253


/tmp/ipykernel_653/4201104216.py:19: UserWarning: Argument(s) 'quality_lower, quality_upper' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=70, quality_upper=100, p=0.3),
/tmp/ipykernel_653/4201104216.py:20: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 189MB/s]


open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B / 1.71GB            

...

STAGE1 Epoch 6/10:   0%|          | 0/1438 [00:00<?, ?it/s]

/tmp/ipykernel_653/3714771439.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=cfg.use_amp):


STAGE1 | Ep  6/10 | Loss: 0.3194 | AUC: 0.9982 | PR-AUC: 0.9979


STAGE1 Epoch 7/10:   0%|          | 0/1438 [00:00<?, ?it/s]

/tmp/ipykernel_653/3714771439.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=cfg.use_amp):


In [9]:
# ============================================================
# Cell 8 (Modified): Finish Line & Final Evaluation
# ============================================================
import os
import csv
import glob
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, roc_auc_score

def load_manifest(path):
    with open(path) as f: return list(csv.DictReader(f))

val_manifest = load_manifest(VAL_MANIFEST)
test_manifest = load_manifest(TEST_MANIFEST)

val_ds = FaceAuthenticityDataset(val_manifest, VAL_DIR, train=False)
test_ds = FaceAuthenticityDataset(test_manifest, TEST_DIR, train=False)

pin_memory = torch.cuda.is_available()
persistent = cfg.num_workers > 0
prefetch = 2 if persistent else None

val_loader = DataLoader(
    val_ds, batch_size=cfg.batch_size, shuffle=False,
    num_workers=cfg.num_workers, pin_memory=pin_memory,
    persistent_workers=persistent, prefetch_factor=prefetch
)

test_loader = DataLoader(
    test_ds, batch_size=cfg.batch_size, shuffle=False,
    num_workers=cfg.num_workers, pin_memory=pin_memory,
    persistent_workers=persistent, prefetch_factor=prefetch
)

# 1. Build Model
model = FusionModel(cfg).to(DEVICE)

# 2. Find the best Stage 2 checkpoint manually
stage2_checkpoints = glob.glob(os.path.join(cfg.checkpoint_dir, "stage2_epoch_*_auc_*.pt"))
if not stage2_checkpoints:
    # Fallback to the latest resume checkpoint if the best tracker failed
    best_ckpt = os.path.join(cfg.checkpoint_dir, "latest_training_checkpoint.pt")
else:
    # Sort by AUC in the filename to get the best one
    stage2_checkpoints.sort()
    best_ckpt = stage2_checkpoints[-1]

print(f"\n--- SKIPPING TRAINING ---")
print(f"Loading best Stage 2 weights from: {os.path.basename(best_ckpt)}")
checkpoint = torch.load(best_ckpt, map_location=DEVICE)
if "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)
model.eval()

print("\n--- FINAL EVALUATION (TTA & Temperature Scaling) ---")

print("1. Calibrating Temperature on Validation Set...")
val_logits, val_labels = collect_predictions(model, val_loader, use_tta_flip=cfg.use_tta_flip)
scaler_model = TemperatureScaler().to(DEVICE)
optimal_temp = scaler_model.fit(val_logits.to(DEVICE), val_labels.float().to(DEVICE))
print(f"Optimal Temperature Calibrated: {optimal_temp:.4f}")

print("\n2. Evaluating on Unseen Test Set...")
test_logits, test_labels = collect_predictions(model, test_loader, use_tta_flip=cfg.use_tta_flip)

# Scale Test Predictions using Validation-learned Temperature
y_prob = torch.sigmoid(test_logits / optimal_temp).numpy()
y_true = test_labels.numpy()
preds = (y_prob >= 0.5).astype(int)

print("\n--- FINAL TEST REPORT ---")
print(classification_report(y_true, preds, target_names=['Real', 'AI']))
print(f"ROC-AUC: {roc_auc_score(y_true, y_prob):.4f}")

# Save Final Artifacts
final_path = os.path.join(cfg.checkpoint_dir, "best_final_model.pt")
torch.save(model.state_dict(), final_path)
print(f"\nArtifacts successfully saved to {final_path}")


/tmp/ipykernel_1357/893107435.py:19: UserWarning: Argument(s) 'quality_lower, quality_upper' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=70, quality_upper=100, p=0.3),
/tmp/ipykernel_1357/893107435.py:20: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 177MB/s]


open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B / 1.71GB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(



--- SKIPPING TRAINING ---
Loading best Stage 2 weights from: stage2_epoch_04_auc_0.9998.pt

--- FINAL EVALUATION (TTA & Temperature Scaling) ---
1. Calibrating Temperature on Validation Set...
Optimal Temperature Calibrated: 0.7712

2. Evaluating on Unseen Test Set...

--- FINAL TEST REPORT ---
              precision    recall  f1-score   support

        Real       1.00      0.99      0.99      1597
          AI       0.99      0.99      0.99      1274

    accuracy                           0.99      2871
   macro avg       0.99      0.99      0.99      2871
weighted avg       0.99      0.99      0.99      2871

ROC-AUC: 0.9999

Artifacts successfully saved to /content/drive/MyDrive/AI_Image_Detection/checkpoints_final/best_final_model.pt
